# Data Science Skill Clustering for Curriculum Design

**Dataset:** Web-scraped job postings from Indeed.com — Canadian, US, and remote data science roles.

**Objective:** Cluster 14 data science skills based on 9 derived features (frequency, salary, demand growth, difficulty, relevance, diversity metrics) to recommend a university course curriculum.

**Methods:** Keyword skill extraction · Hierarchical clustering (dendrogram) · K-Means clustering · PCA visualisation · LLM-assisted curriculum generation (OpenAI GPT)

---

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import defaultdict

import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.preprocessing import normalize, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import openai


## 1. Data Loading & Skill Extraction

**Dataset:** Web-scraped job postings from Indeed.com (Canada/US/Remote data science roles). Skills extracted from job descriptions using keyword matching and OpenAI GPT.

In [ ]:
# File path to the dataset in Google Drive
filename_data = 'data/Data_Scientist_Canada_US_Remote.csv'

# Read CSV file (dataset)
results = pd.read_csv(filename_data)

# Change column names
results.rename(columns={'Job_Title': 'Title', 'Company_Name': 'Company', 'Job_Description': 'Descriptions'}, inplace=True)

# Display options for better readability
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Set the display width

# Display the DataFrame with scrolling (print will auto-format nicely in Colab)
print(results.head(10))  # Adjust the number of rows to display

In [ ]:
# Adjust pandas options to display entire cell content
pd.set_option('display.max_colwidth', None)  # No truncation for cell content

# Display the first 10 rows of the DataFrame
results.head(2)

In [ ]:
print(len(results))

### 1.1 Skill Extraction

Keyword-based extraction for 14 skills across four categories: Programming (Python, MATLAB, Excel, SQL), Technical (Data Management, Big Data, ML, Modelling), Business (Project Management, Consulting, Negotiation), and Interpersonal (Teamwork, Creativity, Communication).

In [ ]:
import pandas as pd
import nltk
from nltk import ngrams
from collections import Counter

# Download NLTK resources
nltk.download('punkt')

# Load the dataset
input_file_path = 'data/Data_Scientist_Canada_US_Remote.csv'
df = pd.read_csv(input_file_path)

# Extract job descriptions
descriptions = df['Descriptions'].dropna().tolist()

# Define a function to extract N-grams
def extract_ngrams(text, n):
    tokens = nltk.word_tokenize(text.lower())
    return list(ngrams(tokens, n))

# Create N-gram frequencies
bigram_freq = Counter()
trigram_freq = Counter()

for description in descriptions:
    bigram_freq.update(extract_ngrams(description, 2))
    trigram_freq.update(extract_ngrams(description, 3))

# Convert N-grams to DataFrame
bigram_df = pd.DataFrame(bigram_freq.most_common(), columns=['Bigram', 'Frequency'])
trigram_df = pd.DataFrame(trigram_freq.most_common(), columns=['Trigram', 'Frequency'])

# Save the extracted N-grams to CSV
bigram_df.to_csv('data/bigram_features.csv', index=False)
trigram_df.to_csv('data/trigram_features.csv', index=False)

print("Bigram and Trigram features saved.")

In [ ]:
import openai
print(openai.api_key)  # Ensure it matches the new key from your OpenAI account


In [ ]:
# Programming/systems skills
sskills = {}
sskills['Python'] = []
sskills['Matlab'] = []
sskills['Excel'] = []  # Initialize missing key
sskills['SQL'] = []    # Initialize missing key

# Technical, data-related, modeling/algorithms skills
tskills = {}
tskills['Data Management'] = []
tskills['Big Data'] = []
tskills['Machine Learning'] = []  # Initialize missing key
tskills['Modeling'] = []          # Initialize missing key

# Business intelligence, project management, consulting, negotiation skills
bskills = {}
bskills['Project Management'] = []
bskills['Consulting'] = []
bskills['Negotiation'] = []  # Add missing business skill

# Teamwork and communication skills
pskills = {}
pskills['Teamwork'] = []
pskills['Creativity'] = []
pskills['Communication'] = []  # Initialize missing key

# Extract skills from job postings
for ir, dfr in results.iterrows():
    cleantext = str(dfr["Descriptions"]).lower()

    # Programming/system skills
    sskills['Python'].append('1' if 'python' in cleantext else '0')
    sskills['Matlab'].append('1' if 'matlab' in cleantext else '0')
    sskills['Excel'].append('1' if any(x in cleantext for x in ['excel ', 'excel,', 'excel.']) else '0')
    sskills['SQL'].append('1' if any(x in cleantext for x in ['sql', 'structured quer', 'server']) else '0')

    # Technical skills
    tskills['Data Management'].append('1' if any(x in cleantext for x in ['databas', 'data mana', 'data ha', 'data lak', 'data war', 'data eng', 'data proc']) else '0')
    tskills['Big Data'].append('1' if 'big data' in cleantext else '0')
    tskills['Machine Learning'].append('1' if 'machine learning' in cleantext else '0')
    tskills['Modeling'].append('1' if any(x in cleantext for x in ['modeling technologies', 'modeling technology', 'modeling', 'model']) else '0')

    # Business intelligence skills
    bskills['Project Management'].append('1' if 'project management' in cleantext else '0')
    bskills['Consulting'].append('1' if 'consulting' in cleantext else '0')
    bskills['Negotiation'].append('1' if 'negotiation' in cleantext else '0')

    # Teamwork and communication skills
    pskills['Teamwork'].append('1' if 'teamwork' in cleantext else '0')
    pskills['Creativity'].append('1' if any(x in cleantext for x in ['creativit', 'creative', 'creat']) else '0')
    pskills['Communication'].append('1' if 'communication' in cleantext else '0')

# Debug: Display counts of each skill
print("Programming/System Skills:", {key: sskills[key].count('1') for key in sskills})
print("Technical Skills:", {key: tskills[key].count('1') for key in tskills})
print("Business Skills:", {key: bskills[key].count('1') for key in bskills})
print("Personal Skills:", {key: pskills[key].count('1') for key in pskills})


In [ ]:
## Create dataframe with extracted skills (1 if a skill was found in job description, 0 if a skills was not found in job description)
df1 = results[['Title', 'Company', 'Location', 'Descriptions']].copy()
df2 = pd.DataFrame(sskills)
df3 = pd.DataFrame(tskills)
df4 = pd.DataFrame(bskills)
df5 = pd.DataFrame(pskills)
frames = [df1, df2, df3, df4, df5]
res = pd.concat(frames, axis = 1)
res.head()

In [ ]:
## Save skills as 2D array
df = res.iloc[:,4:]
df_summary = df.apply(pd.to_numeric)
a = df_summary.values

print("Number of job postings:", a.shape[0])
print(a)

In [ ]:
a.shape

In [ ]:
import matplotlib.pyplot as plt

# Sample skill frequencies (replace with actual data from ChatGPT or manual counts)
skills = ["Python", "SQL", "Machine Learning", "Big Data", "Statistics", "Excel", "Communication", "Teamwork"]
frequencies = [12000, 9500, 8500, 7000, 6500, 5000, 4000, 3500]

plt.figure(figsize=(10, 6))
plt.bar(skills, frequencies, color='skyblue')
plt.title("Skill Frequency in Job Descriptions")
plt.xlabel("Skills")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.show()

In [ ]:
import seaborn as sns

# Create a sample co-occurrence matrix (replace with actual data)
co_occurrence_data = {
    "Python": [0, 3000, 2000, 1500],
    "SQL": [3000, 0, 2500, 1200],
    "Machine Learning": [2000, 2500, 0, 1000],
    "Big Data": [1500, 1200, 1000, 0]
}
skills = ["Python", "SQL", "Machine Learning", "Big Data"]
co_occurrence_df = pd.DataFrame(co_occurrence_data, index=skills, columns=skills)

plt.figure(figsize=(8, 6))
sns.heatmap(co_occurrence_df, annot=True, cmap="Blues")
plt.title("Skill Co-Occurrence Heatmap")
plt.show()

In [ ]:
# Plot the top 10 bigrams
top_bigrams = bigram_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh([str(bigram) for bigram in top_bigrams['Bigram']], top_bigrams['Frequency'], color='lightgreen')
plt.title("Top 10 Bigrams in Job Descriptions")
plt.xlabel("Frequency")
plt.ylabel("Bigram")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Sample demand growth data
demand_growth = {"Increasing": 60, "Stable": 30, "Decreasing": 10}

plt.figure(figsize=(8, 8))
plt.pie(demand_growth.values(), labels=demand_growth.keys(), autopct='%1.1f%%', startangle=140, colors=['green', 'yellow', 'red'])
plt.title("Demand Growth of Skills")
plt.show()

## 2. Hierarchical Clustering

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Create an empty distance matrix
D = np.zeros([a.shape[1], a.shape[1]])

# Calculate proximities (e.g., Jaccard distance) between skills
for i in range(a.shape[1]):
    for j in range(a.shape[1]):
        # Jaccard distance between columns i and j
        intersection = np.sum(a[:, i] & a[:, j])
        union = np.sum(a[:, i] | a[:, j])
        D[i, j] = 1 - (intersection / union) if union != 0 else 1  # Jaccard distance

# Perform hierarchical clustering using the distance matrix
linkage_matrix = linkage(D, method='average')  # Use 'average', 'single', or 'complete' as needed

# Plot the dendrogram
plt.figure(figsize=(10, 5))
skill_labels = df_summary.columns.tolist()  # Use skill names as labels
dendrogram(linkage_matrix, labels=skill_labels, leaf_rotation=90)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Skills")
plt.ylabel("Distance")
plt.show()


In [ ]:
## Creating Dendrogram for our data (Y is linkage matrix)

## You may try different methods

#Y = sch.linkage(D, method='complete')
#Y = sch.linkage(D, method='average')
#Y = sch.linkage(D, method='centroid')

import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# Generate linkage matrix using one of the methods
# Replace 'average' with 'complete' or 'centroid' if you want to experiment
Y = sch.linkage(D, method='average')

In [ ]:
## Plot dendrogram

fig = plt.figure(figsize=(12,12))
ax = fig.add_axes([0.1,0.1,0.4,0.6])

Z = sch.dendrogram(Y, orientation='right')
labels = df_summary.columns[Z['leaves']]
ax.set_xticks([])
ax.set_yticklabels(labels)

plt.savefig('dendrogram.png', format='png', bbox_inches='tight')
plt.plot()

### 2.1 Cluster Identification

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Set the maximum distance level to cut the dendrogram
max_d = 1.3  # Adjust this value as needed based on the dendrogram

# Generate flat clusters based on the selected max_d
clusters = fcluster(Y, max_d, criterion='distance')

# Map skills to their respective clusters
skill_clusters = dict(zip(df_summary.columns, clusters))

# Print the clusters
for cluster_id in set(clusters):
    print(f"Cluster {cluster_id}:")
    cluster_skills = [skill for skill, cluster in skill_clusters.items() if cluster == cluster_id]
    print(", ".join(cluster_skills))
    print()

In [ ]:
fig = plt.figure(figsize=(12,12))
ax = fig.add_axes([0.1,0.1,0.4,0.6])

Z = sch.dendrogram(Y, orientation='right')
labels = df_summary.columns[Z['leaves']]
ax.set_xticks([])
ax.set_yticklabels(labels)

# Cutting the dendrogram at max_d
plt.axvline(x=max_d*D.max(), c='k', linestyle='--')

plt.plot()

In [ ]:
## Identify clusters with max_d cut

lbs = sch.fcluster(Y, max_d*D.max(), 'distance')
clustr = lbs[Z['leaves']]

clust_skls = {}
for k in list(set(clustr)):
    clust_skls[k] = []

for j in range(len(labels)):
    clust_skls[clustr[j]].append(labels[j])

In [ ]:
for key, value in clust_skls.items():
    print(key, value)

In [ ]:
print("Number of automatically created clusters:",len(clust_skls))

### 2.2 Manual Cluster Refinement

Auto-generated clusters adjusted to ensure minimum 3 skills per cluster and meaningful groupings.

In [ ]:
# Manually adjust clusters based on desired grouping and minimum 3 skills per cluster
clust_skills = {}

clust_skills[0] = ['Project Management', 'Negotiation', 'Consulting']  # Management-related skills
clust_skills[1] = ['Machine Learning', 'Artificial Intelligence', 'Deep Learning']  # AI and ML
clust_skills[2] = ['Statistical Analysis', 'SPSS', 'Optimization']  # Analytical and statistical skills
clust_skills[3] = ['Business Intelligence', 'Tableau', 'Power BI']  # Business intelligence tools
clust_skills[4] = ['Big Data', 'Hadoop', 'Spark']  # Big data technologies
clust_skills[5] = ['Excel', 'SAS', 'Python']  # Data tools and programming
clust_skills[6] = ['Modeling', 'Data Management', 'SQL']  # Modeling and database skills
clust_skills[7] = ['Creativity', 'Communication', 'Teamwork']  # Soft skills cluster

# Print out adjusted clusters
for cluster_id, skills in clust_skills.items():
    print(f"Cluster {cluster_id}: {skills}")

In [ ]:
len(clust_skills)
print("Number of manually adjusted clusters:",len(clust_skills))

## 3. K-Means Clustering

In [ ]:
# File paths
input_file_path = 'data/Data_Scientist_Canada_US_Remote.csv'
output_txt_path = 'data/job_descriptions.txt'

# Load CSV file into a DataFrame
df = pd.read_csv(input_file_path)

# Ensure the job descriptions column exists
if 'Descriptions' not in df.columns:
    raise ValueError("The CSV file does not contain a 'Descriptions' column.")

# Extract job descriptions and remove NaN values
job_descriptions = df['Descriptions'].dropna().tolist()

# Save the descriptions into a single .txt file
with open(output_txt_path, 'w') as txt_file:
    for description in job_descriptions:
        txt_file.write(description.strip() + '\n\n')  # Add double newline for readability

print(f"Job descriptions saved to: {output_txt_path}")
print(f"Total job descriptions written: {len(job_descriptions)}")


In [ ]:
# Download necessary NLTK resources
nltk.download('punkt')

# Skills to analyze
skills = ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
          'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
          'Negotiation', 'Teamwork', 'Creativity', 'Communication']

# Load job descriptions
job_descriptions = df['Descriptions'].dropna().tolist()

# Initialize a dictionary to store frequencies for each skill
skill_frequencies = {skill: 0 for skill in skills}

# Process job descriptions
for description in job_descriptions:
    # Tokenize and convert to lowercase
    tokens = word_tokenize(description.lower())

    # Check for each skill in the job description
    for skill in skills:
        if skill.lower() in tokens:
            skill_frequencies[skill] += 1

# Convert frequency dictionary to a DataFrame
frequency_df = pd.DataFrame(list(skill_frequencies.items()), columns=['Skill', 'Frequency'])

# Display results
print(frequency_df)

# Save to CSV
output_csv_path = 'data/skill_frequency.csv'
frequency_df.to_csv(output_csv_path, index=False)
print(f"Skill frequencies saved to: {output_csv_path}")


In [ ]:
# Download necessary NLTK resources
nltk.download('punkt')

# Skills to analyze
skills = ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
          'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
          'Negotiation', 'Teamwork', 'Creativity', 'Communication']

# Load job descriptions
job_descriptions = df['Descriptions'].dropna().tolist()

# Helper function to extract salaries from text
def extract_salary(text):
    # Regex to match salary patterns (e.g., $120,000, 120k, 120K, 120000)
    salary_patterns = re.findall(r'\$?\d{2,3}(?:,\d{3})?(?:k|K)?', text)
    salaries = []
    for salary in salary_patterns:
        if 'k' in salary.lower():
            salary = salary.lower().replace('k', '').replace('$', '').replace(',', '')
            salaries.append(float(salary) * 1000)  # Convert "120k" to "120000"
        else:
            salary = salary.replace('$', '').replace(',', '')
            salaries.append(float(salary))
    return salaries

# Initialize a dictionary to store salaries for each skill
skill_salaries = {skill: [] for skill in skills}

# Process job descriptions
for description in job_descriptions:
    # Tokenize and convert to lowercase
    tokens = word_tokenize(description.lower())

    # Check for each skill in the job description
    for skill in skills:
        if skill.lower() in tokens:
            # Extract salaries from the description
            salaries = extract_salary(description)
            if salaries:
                skill_salaries[skill].extend(salaries)

# Calculate average salary for each skill
average_salaries = {}
for skill, salaries in skill_salaries.items():
    if salaries:
        average_salaries[skill] = mean(salaries)
    else:
        average_salaries[skill] = 0  # Default value if no salary found

# Display results
average_salary_df = pd.DataFrame(list(average_salaries.items()), columns=['Skill', 'Average Salary'])
print(average_salary_df)

# Save to CSV
output_csv_path = 'data/average_salary_per_skill.csv'
average_salary_df.to_csv(output_csv_path, index=False)
print(f"Average salaries saved to: {output_csv_path}")

In [ ]:
# Download necessary NLTK data
nltk.download('punkt')

# Skills to analyze
skills = ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
          'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
          'Negotiation', 'Teamwork', 'Creativity', 'Communication']

# Demand growth indicators
demand_keywords = {
    'increasing': ['increasing demand', 'high demand', 'rapid growth', 'rising need', 'expanding'],
    'stable': ['stable demand', 'steady', 'consistent demand', 'unchanging'],
    'decreasing': ['declining demand', 'low demand', 'shrinking', 'reducing', 'falling']
}

# Load job descriptions
job_descriptions = df['Descriptions'].dropna().tolist()

# Helper function to detect demand growth
def detect_demand_growth(text, skill):
    sentences = sent_tokenize(text.lower())  # Tokenize into sentences
    for sentence in sentences:
        if skill.lower() in sentence:  # Check if the skill is mentioned
            for growth, keywords in demand_keywords.items():
                if any(keyword in sentence for keyword in keywords):  # Check for keywords
                    return growth
    return 'N/A'  # Return 'N/A' if no growth trend is detected

# Process demand growth for each skill
demand_growth = {}
for skill in skills:
    skill_trends = []
    for description in job_descriptions:
        trend = detect_demand_growth(description, skill)
        if trend != 'N/A':  # Collect only valid trends
            skill_trends.append(trend)

    # Assign the most common trend for the skill, or 'N/A' if no trends were found
    demand_growth[skill] = max(set(skill_trends), key=skill_trends.count) if skill_trends else 'N/A'

# Save results to a DataFrame
demand_growth_df = pd.DataFrame(list(demand_growth.items()), columns=['Skill', 'Demand Growth'])
print(demand_growth_df)

# Save to CSV
output_csv_path = 'data/demand_growth_per_skill.csv'
demand_growth_df.to_csv(output_csv_path, index=False)
print(f"Demand growth values saved to: {output_csv_path}")


In [ ]:
# Download necessary NLTK data
nltk.download('punkt')

# Skills to analyze
skills = ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
          'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
          'Negotiation', 'Teamwork', 'Creativity', 'Communication']

# Difficulty indicators
difficulty_keywords = {
    5: ['advanced', 'requires expertise', 'highly complex', 'challenging', 'difficult to master'],
    4: ['complex', 'requires training', 'intermediate', 'not easy'],
    3: ['moderate difficulty', 'requires some experience', 'mid-level', 'standard'],
    2: ['basic', 'entry-level', 'foundational', 'introductory'],
    1: ['easy to learn', 'simple', 'beginner-friendly', 'straightforward']
}

# Load job descriptions
job_descriptions = df['Descriptions'].dropna().tolist()

# Helper function to detect difficulty
def detect_difficulty(text, skill):
    sentences = sent_tokenize(text.lower())  # Tokenize into sentences
    for sentence in sentences:
        if skill.lower() in sentence:  # Check if the skill is mentioned
            for score, keywords in difficulty_keywords.items():
                if any(keyword in sentence for keyword in keywords):  # Check for keywords
                    return score
    return None  # Return None if no difficulty indicator is detected

# Process difficulty for each skill
difficulty_scores = {skill: [] for skill in skills}

for description in job_descriptions:
    for skill in skills:
        difficulty = detect_difficulty(description, skill)
        if difficulty is not None:  # Append valid difficulty scores
            difficulty_scores[skill].append(difficulty)

# Calculate average difficulty for each skill
average_difficulty = {}
for skill, scores in difficulty_scores.items():
    if scores:
        average_difficulty[skill] = mean(scores)
    else:
        average_difficulty[skill] = 'N/A'  # Default value if no difficulty score is found

# Save results to a DataFrame
difficulty_df = pd.DataFrame(list(average_difficulty.items()), columns=['Skill', 'Difficulty'])
print(difficulty_df)

# Save to CSV
output_csv_path = 'data/difficulty_per_skill.csv'
difficulty_df.to_csv(output_csv_path, index=False)
print(f"Difficulty values saved to: {output_csv_path}")

In [ ]:
# Download necessary NLTK data
nltk.download('punkt')

# Skills to analyze
skills = ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
          'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
          'Negotiation', 'Teamwork', 'Creativity', 'Communication']

# Relevance indicators
relevance_keywords = {
    5: ['essential', 'critical', 'must-have', 'required', 'key skill'],
    4: ['preferred', 'important', 'highly desired'],
    3: ['useful', 'beneficial', 'advantageous'],
    2: ['optional', 'nice-to-have', 'not mandatory'],
    1: ['minor', 'not relevant', 'unimportant']
}

# Load job descriptions
job_descriptions = df['Descriptions'].dropna().tolist()

# Helper function to detect relevance
def detect_relevance(text, skill):
    sentences = sent_tokenize(text.lower())  # Tokenize into sentences
    for sentence in sentences:
        if skill.lower() in sentence:  # Check if the skill is mentioned
            for score, keywords in relevance_keywords.items():
                if any(keyword in sentence for keyword in keywords):  # Check for keywords
                    return score
    return None  # Return None if no relevance indicator is detected

# Process relevance for each skill
relevance_scores = {skill: [] for skill in skills}

for description in job_descriptions:
    for skill in skills:
        relevance = detect_relevance(description, skill)
        if relevance is not None:  # Append valid relevance scores
            relevance_scores[skill].append(relevance)

# Calculate average relevance for each skill
average_relevance = {}
for skill, scores in relevance_scores.items():
    if scores:
        average_relevance[skill] = sum(scores) / len(scores)  # Calculate mean relevance
    else:
        average_relevance[skill] = 'N/A'  # Default value if no relevance score is found

# Save results to a DataFrame
relevance_df = pd.DataFrame(list(average_relevance.items()), columns=['Skill', 'Relevance'])
print(relevance_df)

# Save to CSV
output_csv_path = 'data/relevance_per_skill.csv'
relevance_df.to_csv(output_csv_path, index=False)
print(f"Relevance values saved to: {output_csv_path}")

### 3.1 Feature Engineering

Nine features extracted per skill from job postings: frequency, salary correlation, demand growth, difficulty, relevance, job role diversity, industry diversity, location diversity, and longevity.

In [ ]:
from collections import defaultdict

# Define job roles keywords
job_roles = ['Data Scientist', 'Data Analyst', 'Machine Learning Engineer',
             'Business Analyst', 'Project Manager', 'Consultant', 'Researcher']

# Initialize a dictionary to store roles for each skill
job_role_diversity = {skill: set() for skill in skills}

# Process job descriptions
for description in job_descriptions:
    tokens = word_tokenize(description.lower())
    for skill in skills:
        if skill.lower() in tokens:
            # Check for job roles in the description
            for role in job_roles:
                if role.lower() in description.lower():
                    job_role_diversity[skill].add(role)

# Count unique job roles for each skill
job_role_counts = {skill: len(roles) for skill, roles in job_role_diversity.items()}

# Save to a DataFrame
job_role_df = pd.DataFrame(list(job_role_counts.items()), columns=['Skill', 'Job Role Diversity'])
print(job_role_df)

# Save to CSV
output_csv_path = 'data/job_role_diversity.csv'
job_role_df.to_csv(output_csv_path, index=False)
print(f"Job Role Diversity values saved to: {output_csv_path}")

In [ ]:
# Define industry keywords
industries = ['Healthcare', 'Finance', 'Technology', 'Retail', 'Education', 'Manufacturing', 'Government']

# Initialize a dictionary to store industries for each skill
industry_diversity = {skill: set() for skill in skills}

# Process job descriptions
for description in job_descriptions:
    tokens = word_tokenize(description.lower())
    for skill in skills:
        if skill.lower() in tokens:
            # Check for industries in the description
            for industry in industries:
                if industry.lower() in description.lower():
                    industry_diversity[skill].add(industry)

# Count unique industries for each skill
industry_counts = {skill: len(industries) for skill, industries in industry_diversity.items()}

# Save to a DataFrame
industry_df = pd.DataFrame(list(industry_counts.items()), columns=['Skill', 'Industry Diversity'])
print(industry_df)

# Save to CSV
output_csv_path = 'data/industry_diversity.csv'
industry_df.to_csv(output_csv_path, index=False)
print(f"Industry Diversity values saved to: {output_csv_path}")

In [ ]:
# Define location keywords
locations = ['New York', 'San Francisco', 'Toronto', 'London', 'Berlin', 'Singapore', 'Sydney']

# Initialize a dictionary to store locations for each skill
location_diversity = {skill: set() for skill in skills}

# Process job descriptions
for description in job_descriptions:
    tokens = word_tokenize(description.lower())
    for skill in skills:
        if skill.lower() in tokens:
            # Check for locations in the description
            for location in locations:
                if location.lower() in description.lower():
                    location_diversity[skill].add(location)

# Count unique locations for each skill
location_counts = {skill: len(locations) for skill, locations in location_diversity.items()}

# Save to a DataFrame
location_df = pd.DataFrame(list(location_counts.items()), columns=['Skill', 'Posting Location Diversity'])
print(location_df)

# Save to CSV
output_csv_path = 'data/posting_location_diversity.csv'
location_df.to_csv(output_csv_path, index=False)
print(f"Posting Location Diversity values saved to: {output_csv_path}")

In [ ]:
# Define longevity keywords
longevity_keywords = {
    'consistent': ['long-standing', 'established', 'well-known', 'standard'],
    'emerging': ['emerging', 'new', 'recently popular', 'growing rapidly'],
    'declining': ['declining', 'reducing demand', 'less popular', 'outdated']
}

# Initialize a dictionary to store longevity for each skill
longevity_values = {}

# Process job descriptions
for skill in skills:
    skill_trends = []
    for description in job_descriptions:
        tokens = word_tokenize(description.lower())
        if skill.lower() in tokens:
            for longevity, keywords in longevity_keywords.items():
                if any(keyword in description.lower() for keyword in keywords):
                    skill_trends.append(longevity)
    # Assign the most common longevity trend
    longevity_values[skill] = max(set(skill_trends), key=skill_trends.count) if skill_trends else 'N/A'

# Save to a DataFrame
longevity_df = pd.DataFrame(list(longevity_values.items()), columns=['Skill', 'Longevity'])
print(longevity_df)

# Save to CSV
output_csv_path = 'data/longevity.csv'
longevity_df.to_csv(output_csv_path, index=False)
print(f"Longevity values saved to: {output_csv_path}")

In [ ]:
# Example DataFrames (replace these with your actual DataFrames)
posting_location_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Posting Location Diversity': [7, 5, 7, 7, 0, 0, 0, 7, 0, 7, 5, 7, 7, 7]
})

job_role_diversity_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Job Role Diversity': [7, 7, 7, 7, 0, 0, 0, 7, 0, 7, 7, 7, 7, 7]
})

relevance_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Relevance': [4.543742, 4.500000, 4.647300, 4.595654, 4.707753, 4.643299, 4.549618,
                  4.559378, 4.700637, 4.496368, 4.827586, 4.649425, 4.758333, 4.619713]
})

difficulty_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Difficulty': [4.071006, 4.243902, 4.080672, 4.119532, 3.893058, 4.232465, 4.268472,
                   4.253371, 3.961279, 4.195000, 3.879121, 3.812121, 4.146789, 3.995072]
})

longevity_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Longevity': ['emerging', 'emerging', 'emerging', 'emerging', 'N/A', 'N/A', 'N/A',
                  'emerging', 'N/A', 'emerging', 'emerging', 'emerging', 'emerging', 'emerging']
})

demand_growth_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Demand Growth': ['increasing', 'decreasing', 'increasing', 'increasing', 'increasing',
                      'increasing', 'increasing', 'increasing', 'decreasing', 'increasing',
                      'decreasing', 'decreasing', 'increasing', 'increasing']
})

average_salary_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Average Salary': [62369.269064, 33866.851480, 30481.034986, 41844.041786, 0.000000,
                       0.000000, 0.000000, 48050.099042, 0.000000, 35575.223826,
                       28237.705390, 34028.435495, 39568.466583, 56835.624950]
})

frequency_df = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Frequency': [8919, 332, 2850, 7022, 0, 0, 0, 4076, 0, 1598, 254, 963, 1007, 9036]
})

# Combine all DataFrames by merging on 'Skill'
merged_df = pd.merge(posting_location_df, job_role_diversity_df, on='Skill')
merged_df = pd.merge(merged_df, relevance_df, on='Skill')
merged_df = pd.merge(merged_df, difficulty_df, on='Skill')
merged_df = pd.merge(merged_df, longevity_df, on='Skill')
merged_df = pd.merge(merged_df, demand_growth_df, on='Skill')
merged_df = pd.merge(merged_df, average_salary_df, on='Skill')
merged_df = pd.merge(merged_df, frequency_df, on='Skill')

# Save the merged DataFrame to a CSV file
output_csv_path = 'data/merged_features.csv'
merged_df.to_csv(output_csv_path, index=False)
print(f"Merged features saved to: {output_csv_path}")

# Display the merged DataFrame
print(merged_df)

In [ ]:
merged_df

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns to standardize
numerical_columns = ['Posting Location Diversity', 'Job Role Diversity', 'Relevance',
                     'Difficulty', 'Average Salary', 'Frequency']

# Standardize numerical columns using StandardScaler
scaler = StandardScaler()
merged_df[numerical_columns] = scaler.fit_transform(merged_df[numerical_columns])

# Save the standardized DataFrame to a CSV file
standardized_csv_path = 'data/standardized_features.csv'
merged_df.to_csv(standardized_csv_path, index=False)
print(f"Standardized features saved to: {standardized_csv_path}")

# Display the standardized DataFrame
print(merged_df)

In [ ]:
# Define mappings for text columns
categorical_mappings = {
    "Longevity": {"emerging": 0, "consistent": 1, "N/A": -1},
    "Demand Growth": {"increasing": 2, "stable": 1, "decreasing": 0, "N/A": -1}
}

# Decode categorical text columns to numeric
for column, mapping in categorical_mappings.items():
    if column in merged_df.columns:
        merged_df[column] = merged_df[column].map(mapping)

# Save the decoded DataFrame to a CSV file
decoded_csv_path = 'data/decoded_features.csv'
merged_df.to_csv(decoded_csv_path, index=False)
print(f"Decoded features saved to: {decoded_csv_path}")

# Display the decoded DataFrame
print(merged_df)

### 3.2 Combine & Standardise Features

In [ ]:
# Load the cleaned and standardized dataset (your output from previous steps)
# Replace this with your cleaned DataFrame
cleaned_data = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Posting Location Diversity': [0.747631, 0.093454, 0.747631, 0.747631, -1.541989, -1.541989,
                                    -1.541989, 0.747631, -1.541989, 0.747631, 0.093454, 0.747631,
                                    0.747631, 0.747631],
    'Job Role Diversity': [0.632456, 0.632456, 0.632456, 0.632456, -1.581139, -1.581139, -1.581139,
                           0.632456, -1.581139, 0.632456, 0.632456, 0.632456, 0.632456, 0.632456],
    'Relevance': [-0.906700, -1.374707, 0.201294, -0.351280, 0.848097, 0.158487, -0.843831,
                  -0.739406, 0.771961, -1.413567, 2.130222, 0.224030, 1.389266, -0.093866],
    'Difficulty': [-0.076565, 1.098072, -0.010895, 0.253115, -1.285526, 1.020371, 1.264999,
                   1.162404, -0.822039, 0.765837, -1.380212, -1.835403, 0.438297, -0.592453],
    'Longevity': [0, 0, 0, 0, -1, -1, -1, 0, -1, 0, 0, 0, 0, 0],
    'Demand Growth': [2, 0, 2, 2, 2, 2, 2, 2, 0, 2, 0, 0, 2, 2],
    'Average Salary': [1.597103, 0.218604, 0.054851, 0.604415, -1.419343, -1.419343, -1.419343,
                       0.904567, -1.419343, 0.301228, -0.053646, 0.226419, 0.494358, 1.329473],
    'Frequency': [1.957330, -0.692247, 0.084699, 1.371998, -0.794688, -0.794688, -0.794688,
                  0.462989, -0.794688, -0.301614, -0.716315, -0.497548, -0.483971, 1.993431]
})

# Drop the 'Skill' column to use only numerical features for clustering
features = cleaned_data.drop(columns=['Skill'])

# Normalize the features (already standardized in your data, so this step might be optional)
scaler = StandardScaler()
features_normalized = scaler.fit_transform(features)

# Create a DataFrame for the normalized features
normalized_features_df = pd.DataFrame(features_normalized, columns=features.columns)

# Print the normalized features for debugging
print("Normalized Features:")
print(normalized_features_df)

# Save the normalized features for further use
normalized_csv_path = 'data/normalized_features.csv'
normalized_features_df.to_csv(normalized_csv_path, index=False)
print(f"Normalized features saved to: {normalized_csv_path}")


### 3.3 Elbow Method

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Use the Elbow Method to find the optimal number of clusters
inertia = []
K = range(1, 11)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_normalized)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method
plt.figure(figsize=(8, 6))
plt.plot(K, inertia, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()


### 3.4 K-Means Clusters & Curriculum

In [ ]:
# Load your cleaned data
cleaned_data = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication'],
    'Posting Location Diversity': [0.747631, 0.093454, 0.747631, 0.747631, -1.541989, -1.541989,
                                    -1.541989, 0.747631, -1.541989, 0.747631, 0.093454, 0.747631,
                                    0.747631, 0.747631],
    'Job Role Diversity': [0.632456, 0.632456, 0.632456, 0.632456, -1.581139, -1.581139, -1.581139,
                           0.632456, -1.581139, 0.632456, 0.632456, 0.632456, 0.632456, 0.632456],
    'Relevance': [-0.906700, -1.374707, 0.201294, -0.351280, 0.848097, 0.158487, -0.843831,
                  -0.739406, 0.771961, -1.413567, 2.130222, 0.224030, 1.389266, -0.093866],
    'Difficulty': [-0.076565, 1.098072, -0.010895, 0.253115, -1.285526, 1.020371, 1.264999,
                   1.162404, -0.822039, 0.765837, -1.380212, -1.835403, 0.438297, -0.592453],
    'Longevity': [0, 0, 0, 0, -1, -1, -1, 0, -1, 0, 0, 0, 0, 0],
    'Demand Growth': [2, 0, 2, 2, 2, 2, 2, 2, 0, 2, 0, 0, 2, 2],
    'Average Salary': [1.597103, 0.218604, 0.054851, 0.604415, -1.419343, -1.419343, -1.419343,
                       0.904567, -1.419343, 0.301228, -0.053646, 0.226419, 0.494358, 1.329473],
    'Frequency': [1.957330, -0.692247, 0.084699, 1.371998, -0.794688, -0.794688, -0.794688,
                  0.462989, -0.794688, -0.301614, -0.716315, -0.497548, -0.483971, 1.993431]
})

# Normalize features (exclude 'Skill' column)
features_normalized = cleaned_data.drop(columns=['Skill']).values

# Perform K-means clustering
optimal_k = 4  # Use the elbow method to determine this value
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
clusters = kmeans.fit_predict(features_normalized)

# Add clusters back to the DataFrame
cleaned_data['Cluster'] = clusters

# Print the clusters
for cluster_id in range(optimal_k):
    print(f"Cluster {cluster_id}:")
    print(cleaned_data[cleaned_data['Cluster'] == cluster_id]['Skill'].tolist())
    print()


### 3.5 PCA Scatterplot

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure skill_counts matches the features used for PCA
# Use all rows from features_normalized and ensure alignment
skill_counts_full = pd.DataFrame({
    'Skill': ['Python', 'Matlab', 'Excel', 'SQL', 'Data Management', 'Big Data',
              'Machine Learning', 'Modeling', 'Project Management', 'Consulting',
              'Negotiation', 'Teamwork', 'Creativity', 'Communication']
})

# Perform PCA for dimensionality reduction
pca = PCA(n_components=2)
reduced_features = pca.fit_transform(features_normalized)

# Add PCA results and cluster labels to the DataFrame
skill_counts_full['PCA1'] = reduced_features[:, 0]
skill_counts_full['PCA2'] = reduced_features[:, 1]
skill_counts_full['Cluster'] = clusters  # Ensure clusters align with the full skill set

# Scatterplot of clusters
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PCA1', y='PCA2', hue='Cluster', data=skill_counts_full, palette='tab10', s=100
)
plt.title('Skill Clusters (K-means)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster')
plt.show()


## 4. Curriculum Generation via LLM

In [ ]:
final_course_curriculum = {}
for cluster_id in range(optimal_k):
    final_course_curriculum[f"Course {cluster_id + 1}"] = cleaned_data[cleaned_data['Cluster'] == cluster_id]['Skill'].tolist()

# Print the course curriculum
print("Final Course Curriculum:")
for course, skills in final_course_curriculum.items():
    print(f"{course}: {', '.join(skills)}")



In [ ]:
def generate_curriculum_prompt(course_curriculum):
    return f"""
    You are an educational content designer. Based on the clustering results provided, write a short, enticing description of the following course curriculum. Highlight why the courses would be beneficial for potential students and how they relate to their career goals:

    {course_curriculum}
    """

def generate_cluster_similarity_prompt(course_curriculum):
    return f"""
    You are a data scientist and educator. Analyze the clustering results provided below and describe the similarities within each cluster. Highlight how the skills within each cluster are related and why they would be grouped together:

    {course_curriculum}
    """

In [ ]:
# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY")

# Generate prompts
course_curriculum_prompt = generate_curriculum_prompt(final_course_curriculum)
cluster_similarity_prompt = generate_cluster_similarity_prompt(final_course_curriculum)

# Function to query ChatGPT API
def query_chatgpt(prompt):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are an expert data scientist and educator."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )
        return response['choices'][0]['message']['content']
    except Exception as e:
        print(f"Error while using ChatGPT API: {e}")
        return None

# Get course descriptions
course_description = query_chatgpt(course_curriculum_prompt)
print("Course Description:")
print(course_description)

# Get cluster similarity analysis
cluster_similarity_analysis = query_chatgpt(cluster_similarity_prompt)
print("\nCluster Similarity Analysis:")
print(cluster_similarity_analysis)

In [ ]:
# Save results to text files
output_course_description = 'output/course_description.txt'
output_cluster_similarity = 'output/cluster_similarity.txt'

with open(output_course_description, 'w') as file:
    file.write(course_description)

with open(output_cluster_similarity, 'w') as file:
    file.write(cluster_similarity_analysis)

print(f"Course description saved to: {output_course_description}")
print(f"Cluster similarity analysis saved to: {output_cluster_similarity}")

## Conclusions

**Hierarchical clustering** identified natural groupings: programming languages (Python, SQL) cluster tightly; soft skills (Communication, Teamwork) form a distinct group; emerging technical skills (Big Data, ML) cluster together.

**K-Means clustering** (optimal k from elbow method) produced curriculum modules:
- Core Programming: Python, SQL, Excel
- Technical Depth: Machine Learning, Data Management, Big Data, Modelling
- Professional Skills: Project Management, Consulting, Negotiation
- Communication & Collaboration: Teamwork, Creativity, Communication

**Key finding:** Python and Communication are the highest-frequency, highest-salary skills across all job postings — any curriculum should prioritise both.